# Census Income Classification with PyTorch

Binary classification: predict whether annual income is **greater than $50,000**.

## Dataset
Place `income.csv` in the repository root (or update `DATA_PATH` below). The assignment's source file is [Google Drive – income.csv](https://drive.google.com/file/d/1ay5vOv2YiOwjKIWnXFT6sptW0gqew0oa/view?usp=sharing).

This notebook detects the target column from common names. If automatic detection is not correct, set `TARGET_COLUMN` manually.

In [1]:
import random\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import TensorDataset, DataLoader\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix\n

SyntaxError: unexpected character after line continuation character (707045405.py, line 1)

In [ ]:
# Reproducibility\nSEED = 42\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntorch.manual_seed(SEED)\nif torch.cuda.is_available():\n    torch.cuda.manual_seed_all(SEED)\n\nDEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nDATA_PATH = '../income.csv'  # from notebooks/; change if your CSV is elsewhere\nTARGET_COLUMN = None        # e.g. 'income'; None enables automatic detection\nprint('Device:', DEVICE)\n

In [ ]:
# Load and inspect data\ndf = pd.read_csv(DATA_PATH)\ndf.columns = [str(c).strip() for c in df.columns]\nprint('Shape:', df.shape)\ndisplay(df.head())\nprint(df.dtypes)\n

## 1. Data preparation
Identify the label, then separate categorical and continuous predictors. The target is normalized to two classes: `<=50K` (0) and `>50K` (1).

In [ ]:
if TARGET_COLUMN is None:\n    candidates = ['income', 'salary', 'annual_income', 'label', 'target', 'class']\n    matches = [c for c in df.columns if c.lower().strip().replace(' ', '_') in candidates]\n    if not matches:\n        # Common Adult/Census dataset target column is often the final column.\n        matches = [df.columns[-1]]\n    target_col = matches[0]\nelse:\n    target_col = TARGET_COLUMN\nassert target_col in df.columns, f'Target column {target_col!r} not found.'\nprint('Target column:', target_col)\n\ny_raw = df[target_col].astype(str).str.strip().str.rstrip('.')\npositive_values = {'>50k', '1', 'yes', 'true', 'high', '> $50,000', 'above 50k'}\nunique_labels = sorted(y_raw.unique().tolist())\nif len(unique_labels) != 2:\n    raise ValueError(f'Expected a binary target, found {len(unique_labels)} classes: {unique_labels}')\npositive_label = next((v for v in unique_labels if v.lower().replace(' ', '') in positive_values), None)\nif positive_label is None:\n    positive_label = next((v for v in unique_labels if '>' in v or '50k' in v.lower() and '>' in v), unique_labels[-1])\ny = (y_raw == positive_label).astype('int64').to_numpy()\nprint('Label mapping:', {v: int(v == positive_label) for v in unique_labels})\n\nX = df.drop(columns=[target_col]).copy()\n# Treat numeric columns as continuous; remaining columns are categorical.\ncontinuous_cols = X.select_dtypes(include=[np.number]).columns.tolist()\ncategorical_cols = [c for c in X.columns if c not in continuous_cols]\nprint('Categorical columns:', categorical_cols)\nprint('Continuous columns:', continuous_cols)\n\n# Missing-value handling and categorical integer codes.\nX_cat = X[categorical_cols].copy()\nfor col in categorical_cols:\n    X_cat[col] = X_cat[col].fillna('Unknown').astype(str).str.strip()\n    X_cat[col] = pd.Categorical(X_cat[col]).codes\nX_cont = X[continuous_cols].copy()\nfor col in continuous_cols:\n    X_cont[col] = pd.to_numeric(X_cont[col], errors='coerce')\n    X_cont[col] = X_cont[col].fillna(X_cont[col].median())\n\nX_cat_np = X_cat.to_numpy(dtype=np.int64)\nX_cont_np = X_cont.to_numpy(dtype=np.float32)\nprint('Categorical array:', X_cat_np.shape, 'Continuous array:', X_cont_np.shape, 'Labels:', y.shape)\n

In [ ]:
# Exact 25,000 / 5,000 split (as specified). Stratification preserves class proportions.\nif len(df) != 30000:\n    print(f'Note: dataset has {len(df):,} rows; this notebook still uses a 25,000/5,000 split if possible.')\nif len(df) < 30000:\n    raise ValueError('The assignment requires at least 30,000 rows for a 25,000/5,000 split.')\nindices = np.arange(len(df))\ntrain_idx, test_idx = train_test_split(indices, train_size=25000, test_size=5000, random_state=SEED, stratify=y)\n\nX_cat_train, X_cat_test = X_cat_np[train_idx], X_cat_np[test_idx]\nX_cont_train, X_cont_test = X_cont_np[train_idx], X_cont_np[test_idx]\ny_train, y_test = y[train_idx], y[test_idx]\n\n# Convert arrays to PyTorch tensors.\nX_cat_train_t = torch.tensor(X_cat_train, dtype=torch.long)\nX_cat_test_t = torch.tensor(X_cat_test, dtype=torch.long)\nX_cont_train_t = torch.tensor(X_cont_train, dtype=torch.float32)\nX_cont_test_t = torch.tensor(X_cont_test, dtype=torch.float32)\ny_train_t = torch.tensor(y_train, dtype=torch.long)\ny_test_t = torch.tensor(y_test, dtype=torch.long)\nprint('Train:', len(y_train_t), 'Test:', len(y_test_t))\n

## 2. Model design
Each categorical column gets an embedding. Continuous features are batch-normalized. The combined representation passes through one 50-neuron hidden layer with ReLU and dropout (`p=0.4`), then produces two class logits.

In [ ]:
embedding_sizes = []\nfor i, col in enumerate(categorical_cols):\n    n_categories = int(X_cat_np[:, i].max()) + 1\n    embedding_sizes.append((n_categories, min(50, (n_categories + 1) // 2)))\n\nclass TabularModel(nn.Module):\n    def __init__(self, embedding_sizes, n_cont, hidden_size=50, dropout_p=0.4):\n        super().__init__()\n        self.embeddings = nn.ModuleList([nn.Embedding(n, d) for n, d in embedding_sizes])\n        self.n_cont = n_cont\n        self.bn_cont = nn.BatchNorm1d(n_cont) if n_cont > 0 else None\n        n_emb = sum(d for _, d in embedding_sizes)\n        self.fc1 = nn.Linear(n_emb + n_cont, hidden_size)\n        self.dropout = nn.Dropout(dropout_p)\n        self.output = nn.Linear(hidden_size, 2)\n\n    def forward(self, x_cat, x_cont):\n        embedded = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]\n        x = torch.cat(embedded, dim=1) if embedded else None\n        if self.n_cont > 0:\n            x_cont = self.bn_cont(x_cont)\n            x = torch.cat([x, x_cont], dim=1) if x is not None else x_cont\n        x = torch.relu(self.fc1(x))\n        x = self.dropout(x)\n        return self.output(x)\n\nmodel = TabularModel(embedding_sizes, X_cont_train_t.shape[1]).to(DEVICE)\nprint(model)\n

## 3. Training
Train for 300 epochs using `CrossEntropyLoss` and Adam with learning rate 0.001.

In [ ]:
BATCH_SIZE = 512\ntrain_ds = TensorDataset(X_cat_train_t, X_cont_train_t, y_train_t)\ntrain_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)\ncriterion = nn.CrossEntropyLoss()\noptimizer = torch.optim.Adam(model.parameters(), lr=0.001)\nEPOCHS = 300\nhistory = []\n\nfor epoch in range(EPOCHS):\n    model.train()\n    running_loss = 0.0\n    for cat_batch, cont_batch, labels in train_loader:\n        cat_batch = cat_batch.to(DEVICE)\n        cont_batch = cont_batch.to(DEVICE)\n        labels = labels.to(DEVICE)\n        optimizer.zero_grad()\n        logits = model(cat_batch, cont_batch)\n        loss = criterion(logits, labels)\n        loss.backward()\n        optimizer.step()\n        running_loss += loss.item() * len(labels)\n    epoch_loss = running_loss / len(train_loader.dataset)\n    history.append(epoch_loss)\n    if (epoch + 1) % 25 == 0 or epoch == 0:\n        print(f'Epoch [{epoch+1:3d}/{EPOCHS}] - loss: {epoch_loss:.4f}')\n

## 4. Evaluation
Report test cross-entropy loss and accuracy. The test set is held out from model training.

In [ ]:
model.eval()\nwith torch.no_grad():\n    test_logits = model(X_cat_test_t.to(DEVICE), X_cont_test_t.to(DEVICE))\n    test_loss = criterion(test_logits, y_test_t.to(DEVICE)).item()\n    predictions = test_logits.argmax(dim=1).cpu().numpy()\n\ntest_accuracy = accuracy_score(y_test, predictions)\nprint(f'Test Loss: {test_loss:.4f}')\nprint(f'Test Accuracy: {test_accuracy:.4%}')\nprint('Confusion matrix:\\n', confusion_matrix(y_test, predictions))\nprint(classification_report(y_test, predictions, target_names=['<=50K', '>50K']))\n

## 5. Bonus: predict one new person
Provide a dictionary using the original feature column names. Missing fields are filled using training-data defaults. Categorical values not seen during training map to code 0.

In [ ]:
# Keep category mappings and continuous medians for inference.\ncategory_maps = {}\nfor col in categorical_cols:\n    values = pd.Series(X[col].fillna('Unknown').astype(str).str.strip()).astype('category').cat.categories\n    category_maps[col] = {value: i for i, value in enumerate(values)}\ncontinuous_medians = {c: float(X_cont[c].median()) for c in continuous_cols}\n\ndef predict_income(new_person):\n    \"\"\"Return predicted income class and probabilities for one input dictionary.\"\"\"\n    cat_values = []\n    for col in categorical_cols:\n        value = str(new_person.get(col, 'Unknown')).strip()\n        cat_values.append(category_maps[col].get(value, 0))\n    cont_values = [float(new_person.get(c, continuous_medians[c])) for c in continuous_cols]\n    cat_tensor = torch.tensor([cat_values], dtype=torch.long, device=DEVICE)\n    cont_tensor = torch.tensor([cont_values], dtype=torch.float32, device=DEVICE)\n    model.eval()\n    with torch.no_grad():\n        probabilities = torch.softmax(model(cat_tensor, cont_tensor), dim=1)[0].cpu().numpy()\n    predicted_class = int(np.argmax(probabilities))\n    return {'prediction': '>50K' if predicted_class == 1 else '<=50K',\n            'probability_<=50K': float(probabilities[0]),\n            'probability_>50K': float(probabilities[1])}\n\n# Example: replace keys/values with columns and valid values from your CSV.\nexample_person = {c: X.iloc[0][c] for c in X.columns}\nprint(predict_income(example_person))\n

## Conclusion
The notebook prints the final test loss and accuracy after training. Keep the executed notebook (with outputs) in `notebooks/census_income_workshop.ipynb` before submitting. Results depend on the exact CSV contents and preprocessing.